# Transformer 模型实现（手写版）

我们从零开始写 Transformer。


In [ ]:
import torch
from torch import nn
import math
import torch.nn.functional as F

## 第 1 步：模拟输入 token id

真实文本进入模型之前，会先被 tokenizer 转成 token id。

这里我们先不用真实 tokenizer，只用随机整数模拟一批 token id。为了看得清楚，先用很小的数字。

In [19]:
batch_size = 2   # 一次输入 2 条句子
seq_len    = 6   # 每条句子有 6 个 token
vocab_size = 20  # 假设词表里一共有 20 个 token

src = torch.randint(0, vocab_size, (batch_size, seq_len), dtype=torch.long)

print(f"{src=}")
print(f"{src.shape=}")

src=tensor([[11,  8,  0,  4, 13,  7],
        [ 3, 14,  7, 14, 16, 17]])
src.shape=torch.Size([2, 6])


## 第 2 步：词嵌入

token id 只是编号，模型不能直接理解编号之间的含义。

所以需要用 `nn.Embedding` 把每个 token id 转成一个向量。这个向量的维度叫 `d_model`。

In [20]:
d_model = 8  # 为了方便观察，先让每个 token 变成 8 维向量

encoder_embedding = nn.Embedding(vocab_size, d_model)

src_embedding = encoder_embedding(src)

src.shape, src_embedding.shape

(torch.Size([2, 6]), torch.Size([2, 6, 8]))

上面的形状变化是：

```text
[batch_size, seq_len] -> [batch_size, seq_len, d_model]
[2, 6] -> [2, 6, 8]
```

也就是：每个 token id 都被换成了一个 8 维向量。

## 第 3 步：位置编码

词嵌入只表示“这个 token 是什么”，但它不表示“这个 token 在第几个位置”。

Transformer 没有 RNN 那种天然的时间顺序结构，所以需要额外加入位置信息。

经典位置编码公式：

$$PE(pos, 2i) = sin(pos / 10000^{2i / d_{model}})$$

$$PE(pos, 2i + 1) = cos(pos / 10000^{2i / d_{model}})$$

先不用纠结公式为什么这样设计，先抓住它的作用：

- 为每个位置生成一个向量
- 这个向量维度和词嵌入一样，都是 `d_model`
- 然后把位置向量加到词嵌入上
- 相加前后形状不变

In [24]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100, dropout=0.1):
        """
        d_model: 每个 token 向量的维度，必须和词嵌入输出的最后一维一致
        max_len:  提前准备多少个位置的位置编码
        dropout:  加完位置编码后，随机丢弃一部分特征，防止模型过拟合
        """
        super().__init__()

        # Dropout 本身不改变张量形状，只是在训练时随机把部分值置为 0。
        self.dropout = nn.Dropout(dropout)

        # 创建一个全 0 矩阵，用来存放所有位置的位置编码。
        # 行表示位置：第 0 个 token、第 1 个 token、第 2 个 token……
        # 列表示向量维度：第 0 维、第 1 维、第 2 维……
        # 形状：[max_len, d_model]
        pe = torch.zeros(max_len, d_model)

        # position 表示位置编号。
        # torch.arange(0, max_len) 得到：[0, 1, 2, ..., max_len - 1]
        # unsqueeze(1) 把它从 [max_len] 变成 [max_len, 1]
        # 这样后面可以和 div_term 做广播运算。
        # 形状：[max_len, 1]
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        # div_term 对应公式里的 10000^(2i / d_model) 的倒数部分。
        # 原公式：pos / 10000^(2i / d_model)
        # 这里写成：pos * exp(-log(10000) * 2i / d_model)
        # 两种写法数学上等价，但后者更方便用 PyTorch 实现。
        # torch.arange(0, d_model, 2) 只取偶数维索引：0, 2, 4, 6...
        # 形状：[d_model / 2]
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        # position * div_term 会发生广播：
        # position: [max_len, 1]
        # div_term: [d_model / 2]
        # 结果形状：[max_len, d_model / 2]
        #
        # 偶数维使用 sin，对应公式：PE(pos, 2i)
        # 0::2 表示从第 0 维开始，每隔 2 个维度取一次：0, 2, 4, 6...
        pe[:, 0::2] = torch.sin(position * div_term)

        # 奇数维使用 cos，对应公式：PE(pos, 2i + 1)
        # 1::2 表示从第 1 维开始，每隔 2 个维度取一次：1, 3, 5, 7...
        pe[:, 1::2] = torch.cos(position * div_term)

        # 当前 pe 的形状是 [max_len, d_model]。
        # 但是输入 x 的形状是 [batch_size, seq_len, d_model]。
        # 为了让 pe 能和 x 相加，这里在最前面增加一个 batch 维度。
        # 形状变化：[max_len, d_model] -> [1, max_len, d_model]
        # 之后和 x 相加时，PyTorch 会自动广播到 batch_size。
        pe = pe.unsqueeze(0)

        # register_buffer 的意思是：
        # 1. pe 不是可训练参数，不会被优化器更新
        # 2. pe 会被保存到模型的 state_dict 里
        # 3. 调用 model.to(device) 时，pe 会一起移动到 CPU/GPU
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x 是词嵌入后的输入。
        # 形状：[batch_size, seq_len, d_model]

        # 取出当前输入序列的真实长度。
        # 比如 x.shape = [2, 6, 8]，那么 seq_len = 6。
        seq_len = x.size(1)

        # self.pe 里提前保存了 max_len 个位置编码。
        # 当前输入只需要前 seq_len 个位置编码。
        # self.pe[:, :seq_len, :] 的形状是 [1, seq_len, d_model]
        # x 的形状是 [batch_size, seq_len, d_model]
        # 二者相加后，形状仍然是 [batch_size, seq_len, d_model]
        x = x + self.pe[:, :seq_len, :]

        # 返回加入位置信息后的词向量。
        return self.dropout(x)

In [25]:
position_encoding           = PositionalEncoding(d_model=d_model)

src_embedding_with_position = position_encoding(src_embedding)

src_embedding.shape, src_embedding_with_position.shape

(torch.Size([2, 6, 8]), torch.Size([2, 6, 8]))

In [26]:
# 位置编码本身的形状
position_encoding.pe.shape

torch.Size([1, 100, 8])

到这里，输入已经完成了 Transformer 的最前置处理：

```text
token id -> 词向量 -> 加入位置信息
```

下面开始写注意力机制。

## 第 4 步：缩放点积注意力

注意力机制要解决的问题是：

对于当前位置的 token，应该重点关注序列里的哪些 token？

Transformer 里最基础的注意力计算叫 **Scaled Dot-Product Attention**，公式是：

$$Attention(Q, K, V) = softmax(\frac{QK^T}{\sqrt{d_k}})V$$

先理解三个名字：

- `Q`：Query，查询，表示“我想找什么”
- `K`：Key，键，表示“我有什么特征可以被匹配”
- `V`：Value，值，表示“如果你关注我，就从我这里取走什么信息”

在自注意力里，`Q、K、V` 都来自同一个输入。这里先直接让它们都等于加入位置编码后的词向量。

In [27]:
# 自注意力：Q、K、V 都来自同一个输入
Q = src_embedding_with_position
K = src_embedding_with_position
V = src_embedding_with_position

Q.shape, K.shape, V.shape

(torch.Size([2, 6, 8]), torch.Size([2, 6, 8]), torch.Size([2, 6, 8]))

### 4.1 计算注意力分数

`Q @ K^T` 表示每个 token 和其他所有 token 做相似度计算。

如果输入长度是 `seq_len = 6`，那么每条句子会得到一个 `[6, 6]` 的注意力分数矩阵。

In [ ]:
# K.transpose(-2, -1) 会把最后两个维度交换
# K:                    [batch_size, seq_len, d_model]
# K.transpose(-2, -1):  [batch_size, d_model, seq_len]
# Q @ K^T:              [batch_size, seq_len, seq_len]
scores = torch.matmul(Q, K.transpose(-2, -1))

scores.shape

### 4.2 缩放注意力分数

当 `d_model` 比较大时，点积结果可能会很大。

如果直接送入 `softmax`，容易让概率分布过于极端，所以要除以 $\sqrt{d_k}$。

In [ ]:
d_k = Q.size(-1)

scaled_scores = scores / math.sqrt(d_k)

scaled_scores.shape

### 4.3 softmax 得到注意力权重

`softmax(dim=-1)` 表示在最后一个维度上做归一化。

也就是说：对于每个 token，它对整句话所有 token 的关注权重加起来等于 1。

In [ ]:
attention_weights = F.softmax(scaled_scores, dim=-1)

attention_weights.shape

In [ ]:
# 检查第一条句子里，每个 token 的注意力权重是否都加起来等于 1
attention_weights[0].sum(dim=-1)

### 4.4 用注意力权重加权求和

注意力权重表示“该关注谁”。

`attention_weights @ V` 表示根据权重，从 `V` 里汇总信息。

In [ ]:
# attention_weights: [batch_size, seq_len, seq_len]
# V:                 [batch_size, seq_len, d_model]
# output:            [batch_size, seq_len, d_model]
attention_output = torch.matmul(attention_weights, V)

attention_output.shape

注意力层的输入和输出形状是一样的：

```text
[batch_size, seq_len, d_model] -> [batch_size, seq_len, d_model]
```

区别是：输出中的每个 token 向量，已经融合了其他 token 的信息。

### 4.5 封装成函数

现在把上面的步骤封装成一个函数，后面写多头注意力时会复用。

In [ ]:
def scaled_dot_product_attention(query, key, value, mask=None):
    # query: [batch_size, seq_len, d_k]
    # key:   [batch_size, seq_len, d_k]
    # value: [batch_size, seq_len, d_v]

    d_k = query.size(-1)

    # 1. 计算 Q 和 K 的相似度分数
    scores = torch.matmul(query, key.transpose(-2, -1))

    # 2. 缩放分数，避免 softmax 过于极端
    scores = scores / math.sqrt(d_k)

    # 3. 如果有 mask，把不允许关注的位置设置成一个很小的数
    # 这样 softmax 之后，这些位置的权重会接近 0
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)

    # 4. 把分数转换成注意力权重
    attention_weights = F.softmax(scores, dim=-1)

    # 5. 根据注意力权重，从 value 中汇总信息
    output = torch.matmul(attention_weights, value)

    return output, attention_weights

In [ ]:
attention_output, attention_weights = scaled_dot_product_attention(Q, K, V)

attention_output.shape, attention_weights.shape